**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/russo-2025"], check=True)
    os.chdir("/content/encore/entries/russo-2025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Russo et al. (2025): the optogenetic claims, on the paper's figshare deposit

Russo et al. (2025), doi:10.1038/s41467-025-58717-2. The claims file is `claims/russo-2025.yaml`.
This notebook tests the three optogenetic claims (`data: elsewhere`) on the processed data the
paper deposited on figshare ("Data Mouse Opto", article 28440311 version 1,
doi:10.6084/m9.figshare.28440311.v1), read in place from the zip. The dandiset 000458
(version 0.230317.0039) holds no optogenetic sessions; `russo.py` is imported for the unit
measures shared with the main notebook (`reproduce_russo_2025.ipynb`).

| Claim | Outcome |
|---|---|
| fig2fg-opto-thalamic-inhibition-delays-rebound | not reproduced |
| figs8b-opto-cortical-inhibition-minimal | reproduced |
| fig2h-opto-gaba-suppresses-both | reproduced |

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import opto
import russo

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

started = time.time()
folders = opto.sessions()
SM_TH = russo.SM_TH
U = opto.load_units(folders, lambda a: a in SM_TH or a.startswith("MOs"))
U["region"] = np.where(U.area.str.startswith("MOs"), "MOs", "SM-TH")
U["ntrials"] = U.groupby("folder").trial.transform(lambda s: max((t.max() if len(t) else -1) for t in s) + 1)
print(f"figshare {opto.FIGSHARE_DOI}: {len(folders)} protocol folders, {len(U)} unit and protocol rows read "
      f"in {time.time() - started:.0f} s")

## Data check

In [ ]:
print("Protocol folders by mouse, session and light target:")
print(folders.groupby(["mouse", "session", "light"]).agg(probes=("probe", "nunique"), protocols=("protocol", lambda s: sorted(set(s)))).to_string())
print("\nTrials per protocol folder (largest trial index + 1):")
print(U.groupby(["light", "kind"]).ntrials.describe()[["min", "50%", "max"]].to_string())
print("\nUnit duration (the metrics' `duration`, ms): min %.3f, median %.3f, max %.3f" % (U.duration.min(), U.duration.median(), U.duration.max()))
print("Area taken from the table or from the peak channel:", U.drop_duplicates(["folder", "cluster_id"]).area_from.value_counts().to_dict())
es = U[U.kind == "ES"]
print("\nSM-TH and MOs units in the ES-only folders, by light target and mouse:")
print(es.groupby(["light", "mouse", "region"]).size().unstack(fill_value=0).to_string())
lat_all = np.concatenate([a for a in U.lat if len(a)])
print("Spike latencies span %.2f to %.2f s around the event" % (lat_all.min(), lat_all.max()))

In [ ]:
# The analyses pair a unit across the protocol folders of a probe by cluster_id. Check that the
# folders describe one spike sort: the same cluster_ids, and the same metrics for each.
KEY = ["mouse", "session", "probe", "cluster_id"]
cols = ["peak_channel", "duration", "area", "amplitude", "snr", "firing_rate", "isi_viol", "presence_ratio"]
g = U.groupby(KEY)
print("Units (mouse, session, probe, cluster_id) of SM-TH and MOs:", g.ngroups, "; protocol folders per unit:",
      g.folder.nunique().value_counts().sort_index().to_dict())
same = pd.DataFrame({c: g[c].nunique(dropna=False) == 1 for c in cols})
print("Units whose value is the same in all four folders, by metric:", {c: f"{int(same[c].sum())} of {len(same)}" for c in cols})
ids = U.groupby(["mouse", "session", "probe", "protocol"]).cluster_id.apply(frozenset)
same_ids = ids.groupby(["mouse", "session", "probe"]).apply(lambda s: s.nunique() == 1)
print(f"Probes whose four folders list the same cluster_ids: {int(same_ids.sum())} of {len(same_ids)}")

The deposit has no description; its layout is described in `opto.py`. Each protocol has 120
trials (240 in some light-alone and ES folders of the thalamic sessions). Light over the thalamus
was used in four mice (627041, 670246, 671439, 673553), as the paper states for Figure 2; light
over MOs in five (670246, 671439, 673553, 717476, 717477), where the Figure S8B legend says two
mice. The deposit holds unit spike latencies only, no LFP, so the LFP latency part of
fig2fg-opto-thalamic-inhibition-delays-rebound cannot be tested. Every unit of the two areas has the same
cluster_ids, peak channel, duration, area and quality metrics in the four protocol folders of
its probe, so the folders are cut from one spike sort and a unit is paired across them by
cluster_id (the check is above). The clock of the spike latencies
is checked below by the response to the pulse in the ES-only folders.

In [ ]:
U["responsive"] = [russo.responsive(r.lat, r.trial, np.ones(r.ntrials, bool)) for r in U.itertuples()]
U["cls"] = np.select([(U.region == "MOs") & (U.duration > russo.RS_MIN_MS), (U.region == "SM-TH") & (U.duration > russo.RELAY_MIN_MS)],
                     ["RS", "relay"], "other")


def median_first(r, win):
    lat, _ = russo.first_spikes(r.lat, r.trial, r.ntrials, win)
    return np.nanmedian(lat) if np.isfinite(lat).any() else np.nan


U["rebound"] = [median_first(r, russo.REBOUND) for r in U.itertuples()]
U["initial"] = [median_first(r, (0.002, 0.010)) for r in U.itertuples()]
es = U[U.kind == "ES"]
early = es.groupby("region").apply(lambda d: pd.Series({
    "units": len(d), "rate_2_10ms": np.mean([((r.lat >= 0.002) & (r.lat < 0.010)).sum() / r.ntrials / 0.008 for r in d.itertuples()]),
    "rate_-300_-292ms": np.mean([((r.lat >= -0.3) & (r.lat < -0.292)).sum() / r.ntrials / 0.008 for r in d.itertuples()])}))
print("ES alone: mean rate in 2-10 ms after the pulse and in an 8 ms window 300 ms before it (Hz):")
print(early.round(2).to_string())
print("\nResponsive units (7 SD rule) in the ES-only folders, by class:")
print(es[es.responsive].groupby(["light", "cls"]).size().unstack(fill_value=0).to_string())

## fig2fg-opto-thalamic-inhibition-delays-rebound

> Optogenetic activation of TRN (thalamic inhibition) at 75 to 125 ms or 125 to 175 ms after the
> pulse progressively delays the rebound of SM-TH and MOs units and the late LFP peak (by 58, 8
> and 31 ms, and by 72, 39 and 77 ms).

Units responsive to the pulse alone, relay units in SM-TH and RS units in MOs (as in the main
notebook); the rebound latency of a unit is the median over trials of its first spike in 75 to
250 ms (Results; Methods, Unit analyses). The delay is the latency with light minus the latency
without, per unit; signed-rank tests on the unit latencies for the three protocol pairs. All
units of the two areas are a variant.

In [ ]:
KEY = ["mouse", "session", "probe", "cluster_id"]


def delays(light, classes=("relay", "RS"), responsive_only=True):
    base = U[(U.kind == "ES") & (U.light == light)].set_index(KEY)
    rows, tests = [], []
    for region, cls in [("SM-TH", classes[0]), ("MOs", classes[1])]:
        b = base[(base.region == region) & ((base.cls == cls) if cls else True) & (base.responsive if responsive_only else True)]
        lat = {"ES": b.rebound}
        for ms in [75, 125]:
            o = U[(U.kind == "ES+opto") & (U.light == light) & (U.light_ms == ms)].set_index(KEY).rebound
            lat[f"ES+Opto{ms}"] = o.reindex(b.index)
        L = pd.DataFrame(lat).dropna() * 1e3
        for k in ["ES+Opto75", "ES+Opto125"]:
            dl = L[k] - L["ES"]
            rows.append({"region": region, "protocol": k, "units": len(L), "mice": L.index.get_level_values(0).nunique(),
                         "ES_mean": L["ES"].mean(), "ES_sd": L["ES"].std(), "with_light_mean": L[k].mean(), "with_light_sd": L[k].std(),
                         "delay_mean": dl.mean(), "delay_sd": dl.std()})
        for a, c in [("ES", "ES+Opto75"), ("ES", "ES+Opto125"), ("ES+Opto75", "ES+Opto125")]:
            tests.append({"region": region, "pair": f"{a} vs {c}", "units": len(L), "p_signed_rank": stats.wilcoxon(L[a], L[c]).pvalue,
                          "median_diff_ms": (L[c] - L[a]).median()})
    return pd.DataFrame(rows), pd.DataFrame(tests)


d_th, t_th = delays("TH")
print("Light over the thalamus; responsive relay (SM-TH) and RS (MOs) units, ms:")
print(d_th.round(2).to_string(index=False))
print(t_th.to_string(index=False))
print("Paper: SM-TH 58 +- 39 and 72 +- 53, MOs 8 +- 28 and 39 +- 33 ms")
v_th, vt_th = delays("TH", classes=(None, None), responsive_only=False)
print("\nVariant, all units of the two areas:")
print(v_th.round(2).to_string(index=False))
print(vt_th.to_string(index=False))
d_mouse = []
base = U[(U.kind == "ES") & (U.light == "TH") & U.responsive & U.cls.isin(["relay", "RS"])].set_index(KEY)
for ms in [75, 125]:
    o = U[(U.kind == "ES+opto") & (U.light == "TH") & (U.light_ms == ms)].set_index(KEY).rebound
    dd = ((o.reindex(base.index) - base.rebound) * 1e3).rename("delay").to_frame().join(base[["region"]])
    d_mouse.append(dd.dropna().groupby(["region", "mouse"]).delay.agg(["mean", "size"]).assign(protocol=f"ES+Opto{ms}"))
print("\nDelay by mouse (ms):")
print(pd.concat(d_mouse).round(1).to_string())

# The criterion's unit part as written: the four mean delays within the paper's SDs and all six signed-rank p < 0.05
exp_delay = {("SM-TH", "ES+Opto75"): (58, 39), ("MOs", "ES+Opto75"): (8, 28),
             ("SM-TH", "ES+Opto125"): (72, 53), ("MOs", "ES+Opto125"): (39, 33)}
dm = d_th.set_index(["region", "protocol"]).delay_mean
within = all(abs(dm[k] - v) <= tol for k, (v, tol) in exp_delay.items())
print(f"\nCriterion, unit part: four delays within tolerance {within}; six tests p < 0.05 "
      f"{int((t_th.p_signed_rank < 0.05).sum())} of 6. The LFP part cannot be tested (no LFP in the deposit).")

Result: not reproduced. Light over the thalamus delays the rebound of SM-TH relay units
by 47.0 +- 13.0 ms (75 ms light) and 52.2 +- 51.3 ms (125 ms), and of MOs RS units by 15.1 +- 18.7
and 36.1 +- 29.3 ms, all four within the paper's values. Five of the six signed-rank tests give p
< 0.05; SM-TH 75 against 125 ms does not (p = 0.20, 38 units from 2 mice). The criterion asks that
all six give p < 0.05 and has no clause for a partial result, so a tested part fails; the LFP part,
which cannot be tested because the deposit holds no LFP, cannot change that. All units of the two
areas pass all six tests, but that is a variant and not the criterion's population. Units are
paired across the protocol folders by cluster_id, which the data check shows to be one sort. This
outcome was proposed after the second verification round and agreed by the third.

## figs8b-opto-cortical-inhibition-minimal

> Optogenetic activation of GABAergic neurons in MOs minimally shifts the rebound latency (MOs
> 208, 213, 220 ms; SM-TH 201, 191, 204 ms for ES alone and with light at 75 or 125 ms), unlike
> thalamic inhibition.

The same measures on the sessions with light over MOs. Reproduced when the six means are within
the paper's SDs and the SM-TH shifts are smaller than those of thalamic inhibition for the same
light window (criterion).

In [ ]:
d_ct, t_ct = delays("MOs")
print("Light over MOs; responsive relay (SM-TH) and RS (MOs) units, ms:")
print(d_ct.round(2).to_string(index=False))
print(t_ct.to_string(index=False))
exp = {("MOs", "ES"): (208, 39), ("MOs", "ES+Opto75"): (213, 40), ("MOs", "ES+Opto125"): (220, 44),
       ("SM-TH", "ES"): (201, 36), ("SM-TH", "ES+Opto75"): (191, 32), ("SM-TH", "ES+Opto125"): (204, 36)}
rows = []
for (reg, prot), (v, tol) in exp.items():
    r = d_ct[(d_ct.region == reg) & (d_ct.protocol == ("ES+Opto75" if prot == "ES" else prot))].iloc[0]
    m = r.ES_mean if prot == "ES" else r.with_light_mean
    rows.append({"region": reg, "protocol": prot, "measured": m, "expected": v, "tolerance": tol, "within": abs(m - v) <= tol})
print(pd.DataFrame(rows).round(2).to_string(index=False))
for ms in ["ES+Opto75", "ES+Opto125"]:
    a = d_ct[(d_ct.region == "SM-TH") & (d_ct.protocol == ms)].delay_mean.iloc[0]
    b = d_th[(d_th.region == "SM-TH") & (d_th.protocol == ms)].delay_mean.iloc[0]
    print(f"SM-TH shift {ms}: MOs light {a:.1f} ms against thalamic light {b:.1f} ms; smaller in size: {abs(a) < abs(b)}")
v_ct, _ = delays("MOs", classes=(None, None), responsive_only=False)
print("\nVariant, all units of the two areas:")
print(v_ct.round(2).to_string(index=False))

In [ ]:
base = U[(U.kind == "ES") & (U.light == "MOs") & U.responsive & U.cls.isin(["relay", "RS"])].set_index(KEY)
lat = {"ES": base.rebound}
for ms in [75, 125]:
    lat[f"ES+Opto{ms}"] = U[(U.kind == "ES+opto") & (U.light == "MOs") & (U.light_ms == ms)].set_index(KEY).rebound.reindex(base.index)
L = (pd.DataFrame(lat) * 1e3).join(base[["region"]]).dropna()
pm8 = L.groupby(["region", "mouse"]).agg(units=("ES", "size"), ES=("ES", "mean"), ES_Opto75=("ES+Opto75", "mean"),
                                         ES_Opto125=("ES+Opto125", "mean"))
pm8["shift75"], pm8["shift125"] = pm8.ES_Opto75 - pm8.ES, pm8.ES_Opto125 - pm8.ES
print("Light over MOs, by mouse: mean rebound latency (ms) of responsive relay (SM-TH) and RS (MOs) units")
print(pm8.round(1).to_string())

# Every pair of the light-over-MOs mice: the six means against the paper's values, and the SM-TH shifts against
# those of thalamic light (the Figure S8B legend reports two mice)
from itertools import combinations
th_shift = d_th[d_th.region == "SM-TH"].set_index("protocol").delay_mean
mice8 = sorted(L.index.get_level_values("mouse").unique())
rows = []
for pair in combinations(mice8, 2):
    Lp = L[L.index.get_level_values("mouse").isin(pair)]
    r = {"mice": " + ".join(m.replace("mouse", "") for m in pair)}
    out = []
    for reg in ["MOs", "SM-TH"]:
        g = Lp[Lp.region == reg]
        r[f"{reg}_units"] = len(g)
        for prot in ["ES", "ES+Opto75", "ES+Opto125"]:
            r[f"{reg} {prot}"] = g[prot].mean() if len(g) else np.nan
            v, tol = exp[(reg, prot)]
            if len(g) and abs(g[prot].mean() - v) > tol:
                out.append(f"{reg} {prot}")
    g = Lp[Lp.region == "SM-TH"]
    r["means_outside"] = ", ".join(out) if out else "none"
    r["SM-TH shifts smaller"] = (bool(len(g)) and abs((g["ES+Opto75"] - g.ES).mean()) < abs(th_shift["ES+Opto75"])
                                 and abs((g["ES+Opto125"] - g.ES).mean()) < abs(th_shift["ES+Opto125"]))
    rows.append(r)
pairs8 = pd.DataFrame(rows)
print("\nPairs of mice (SM-TH columns NaN where the pair has no responsive relay units):")
print(pairs8.round(1).to_string(index=False))
ok = (pairs8.means_outside == "none") & pairs8["SM-TH shifts smaller"] & (pairs8["SM-TH_units"] > 0)
print(f"Pairs with SM-TH units: {int((pairs8['SM-TH_units'] > 0).sum())} of {len(pairs8)}; "
      f"all six means within tolerance and SM-TH shifts smaller: {int(ok.sum())}")

Result: reproduced. With light over MOs the six mean rebound latencies (MOs 193.5, 196.9, 198.8
ms; SM-TH 202.3, 191.5, 203.5 ms) are all within the paper's values, and the SM-TH shifts (-10.8
and 1.3 ms) are smaller than with thalamic light (47.0 and 52.2 ms). The small shifts are
significant (p = 1e-13 to 0.026), as the Figure S8B legend also reports. The units come from 3
(SM-TH) and 4 (MOs) mice, where the paper reports 2; 196 of the 206 SM-TH units are from 717477.
By mouse the SM-TH shifts are -27.9 to -10.6 ms (75 ms light; -27.9 from the one unit of 717476) and -11.6 to 1.9 ms (125 ms), and
the MOs shifts -11.5 to 17.7 and -12.8 to 23.9 ms, all smaller in size than the SM-TH shifts with
thalamic light. The paper reports two mice. Over the ten pairs of the five mice (table above),
the SM-TH shifts are smaller than with thalamic light in each of the nine pairs with responsive
relay units, but the six means are all within the paper's values in only six pairs. In three
pairs some means fall outside: 670246 with 671439 (SM-TH means with light 156.9 and 157.9 ms),
671439 with 717476 (SM-TH with light at 125 ms, 162.5 ms), and 671439 with 673553 (all three MOs means, 150.4 to 162.4 ms,
from 673553 alone, and the two SM-TH means with light). 670246 with 673553 has no responsive
relay units. So the outcome depends on which two mice the paper used.

## fig2h-opto-gaba-suppresses-both

> Optogenetic activation of GABAergic neurons in either the thalamus or MOs suppresses firing in
> both SM-TH and MOs units within the light period.

Light-alone trials (`opto_5`, latencies taken as timed from the light onset). Per unit, the rate in
0 to 50 ms against a baseline of equal length, -50 to 0 ms (criterion); signed-rank tests. Units:
the responsive relay and RS units of the delay analysis, and all units of the two areas.

In [ ]:
rows = []
light_only = U[U.kind == "opto"].set_index(KEY)
for light in ["TH", "MOs"]:
    base = U[(U.kind == "ES") & (U.light == light)].set_index(KEY)
    for sel_name, sel in [("responsive relay/RS", base.responsive & base.cls.isin(["relay", "RS"])), ("all units", base.region.notna())]:
        for region in ["SM-TH", "MOs"]:
            idx = base.index[sel & (base.region == region)]
            lo = light_only.reindex(idx).dropna(subset=["ntrials"])
            on = np.array([((r.lat >= 0) & (r.lat < 0.05)).sum() / r.ntrials / 0.05 for r in lo.itertuples()])
            pre = np.array([((r.lat >= -0.05) & (r.lat < 0)).sum() / r.ntrials / 0.05 for r in lo.itertuples()])
            ok = (on + pre) > 0
            w = stats.wilcoxon(on[ok], pre[ok]) if ok.sum() > 1 else None
            rows.append({"light": light, "units": sel_name, "region": region, "n": int(ok.sum()),
                         "mice": lo.index.get_level_values(0).nunique(), "baseline_Hz": pre[ok].mean(),
                         "light_Hz": on[ok].mean(), "lower_during_light": int((on[ok] < pre[ok]).sum()),
                         "p_signed_rank": w.pvalue if w else np.nan})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Timing of the light-alone (opto_5) latencies: mean rate over all units of each area, 10 ms bins
edges = np.arange(-0.2, 0.2001, 0.01)
psth, rise = [], []
for light in ["TH", "MOs"]:
    b = U[(U.kind == "ES") & (U.light == light)].set_index(KEY)
    for region in ["SM-TH", "MOs"]:
        lo = light_only.reindex(b.index[b.region == region]).dropna(subset=["ntrials"])
        h = np.array([np.histogram(r.lat, edges)[0] / r.ntrials / 0.01 for r in lo.itertuples()])
        psth.append(pd.Series(h.mean(0), index=np.round(edges[:-1] * 1e3).astype(int), name=f"{light} light, {region} ({len(lo)} units)"))
        on = np.array([((r.lat >= 0) & (r.lat < 0.05)).sum() / r.ntrials / 0.05 for r in lo.itertuples()])
        pre = np.array([((r.lat >= -0.05) & (r.lat < 0)).sum() / r.ntrials / 0.05 for r in lo.itertuples()])
        for k in np.nonzero(on > pre)[0]:
            r = lo.iloc[k]
            rise.append({"light": light, "region": region, "mouse": lo.index[k][0], "cluster_id": lo.index[k][3], "area": r.area,
                         "duration_ms": r.duration, "rate_-50_0ms": pre[k], "rate_0_50ms": on[k]})
print("Light alone: mean rate (Hz) in 10 ms bins from -200 to +200 ms (bin start, ms):")
print(pd.DataFrame(psth).round(1).to_string())
rise = pd.DataFrame(rise)
print("\nUnits (all units of the two areas) whose rate is higher in 0 to 50 ms than in -50 to 0 ms:")
print(rise.groupby(["light", "region"]).size().to_string())
print("\nThe 15 largest rates in 0 to 50 ms among them:")
print(rise.sort_values("rate_0_50ms", ascending=False).head(15).round(2).to_string(index=False))
print("Of these units, under 0.35 ms:", rise.groupby(["light", "region"]).duration_ms.apply(lambda d: f"{int((d < 0.35).sum())} of {len(d)}").to_dict())
print(f"\nRun time {(time.time() - started) / 60:.1f} min")

Result: reproduced. In light-alone trials the rate in 0 to 50 ms is lower than in the 50 ms
before the light for SM-TH and MOs units, with light over the thalamus (SM-TH 4.60 to 0.20 Hz, 38
of 38 units lower; MOs 3.36 to 1.41 Hz, 112 of 122) and over MOs (SM-TH 4.45 to 3.20 Hz, 169 of
210; MOs 3.41 to 1.68 Hz, 246 of 268), all signed-rank p < 1e-5. All units of the two areas give
the same. The light-alone PSTH supports the assumption that the latencies are timed from the light
onset: the rate of MOs units falls from the 0 to 10 ms bin on (from 3.2 to 2.5 Hz with thalamic
light, then to 0.2 Hz from 50 ms), and the SM-TH rate falls below baseline from 20 ms and rebounds
above it from 100 ms. In all SM-TH units with thalamic light the mean rate barely falls (7.37 to
7.15 Hz) though 201 of 207 units are lower: 6 units rise, 5 of them VPM units under 0.35 ms in
627041 whose rate goes from 20 to 28 Hz to 196 to 387 Hz. The outputs do not show what these units
are: they could be GABAergic units driven by the light, mislocated TRN units, or a light artifact.

## Summary

| Claim | Outcome |
|---|---|
| fig2fg-opto-thalamic-inhibition-delays-rebound | not reproduced (LFP part not testable) |
| figs8b-opto-cortical-inhibition-minimal | reproduced |
| fig2h-opto-gaba-suppresses-both | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The units are those responsive to the pulse
alone (the Methods' 7 SD rule, 10 ms bins) and classed relay or RS by duration; the paper's own
unit counts (118 SM-TH and 155 MOs units from 4 mice for thalamic light) are larger than the 38
and 104 responsive units here. The light-alone latencies are assumed timed from the light onset.

What was found about the deposit. It has no description. One mouse's unit tables (673553)
lack the area column, which is taken here from the probe's per-channel areas at each unit's peak
channel. Light over MOs was used in five mice, where the paper says two. There is no LFP, so the
LFP part of Figure 2G cannot be checked.